In [ ]:
import os
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

In [ ]:
# Define the list of subjects
# plt.rcParams['font.family'] = 'Arial'

subs = ['anatomy','clinical_knowledge','college_biology', 'medical_genetics']
Subs = ['(A) Anatomy','(B) Clinical Knowledge','(C) College Biology', '(D) Medical Genetics']

# Define model names and desired order
original_model_names = ['GPT3.5', 'Alpaca', 'GPT4o-mini', 'LLaMA-1', 'LLaMA-2']
desired_order = ['LLaMA-1', 'LLaMA-2', 'Alpaca', 'GPT3.5', 'GPT4o-mini']
desired_indices = [original_model_names.index(model) for model in desired_order]

# Function to get winning probability matrix in the desired order
def get_winning_prob_matrix(scores):
    num_models = scores.shape[1]
    winning_prob_matrix = np.zeros((num_models, num_models))
    
    for i in range(num_models):
        for j in range(num_models):
            if i != j:
                wins = scores[:, desired_indices[i]] > scores[:, desired_indices[j]]
                winning_prob_matrix[i, j] = np.mean(wins)
                
    return winning_prob_matrix

h = 0.3
dim = 16
tr_num = 100
te_num = [50, 50, 50, 16]

fig, axes = plt.subplots(2, 2, figsize=(9, 7))
# Create a custom lighter colormap
cmap = sns.diverging_palette(245, 15, l=55, as_cmap=True)
# sns.set(font="Nimbus Roman")

for index, sub in enumerate(subs):
    # Read the CSV file
    filename = f'data/out/{subs[index]}_h{h}_dim{dim}_tr{tr_num}_te{te_num[index]}.npz'

    if os.path.exists(filename):
        file = np.load(filename)
        scores = file['estimates']
        
        # Get the winning probability matrix
        winning_prob_matrix = get_winning_prob_matrix(scores)
        # winning_prob_matrix_reorder = winning_prob_matrix[desired_indices,:]
        
        # Plot the heatmap in the correct subplot
        ax = axes[index // 2, index % 2]
        sns.heatmap( winning_prob_matrix, annot=True, cmap=cmap, xticklabels=desired_order, yticklabels=desired_order, center=0.5, cbar_kws={"shrink": 0.8}, ax=ax)
        ax.set_title(f'{Subs[index]}', pad=20) 
        ax.set_xlabel('Model')
        ax.set_ylabel('Model')
        ax.tick_params(axis='x', rotation=30) 

# Adjust the margins and spacing between subplots
plt.subplots_adjust(left=0.1, right=0.9, top=0.9, bottom=0.1, wspace=0.8, hspace=0.8)

# plt.tight_layout()
plt.savefig(f'data/out/pic_h{h}_dim{dim}.pdf', dpi=300, format='pdf')
plt.show()

# np.savez(f'data/out/{subs[i]}_h{h}_dim{dim}.npz', **result_dict)

In [ ]:
import numpy as np
import json
import matplotlib.pyplot as plt
import seaborn as sns
from matplotlib.colors import LinearSegmentedColormap


# Assuming subs and te_num are defined earlier
subs = ['anatomy','clinical_knowledge', 'college_biology','medical_genetics']
te_num = [50, 50, 50, 16]

# Adjust the figsize to make the subplots smaller
fig, axes = plt.subplots(1, 4, figsize=(14,3.5))  # Adjusted figure size for smaller subplots

# Define the desired order of model names
desired_order = ['LLaMA-1', 'LLaMA-2', 'Alpaca', 'GPT3.5', 'GPT4o-mini']

# Create a custom colormap with two colors for 0 and 100
cmap = sns.diverging_palette( 15,240, l=55, as_cmap=True)


for i, sub in enumerate(subs):
    # Read the JSON file
    json_path = f'data/eval/{subs[i]}_num=100.json'
    with open(json_path, 'r') as json_file:
        out_data = json.load(json_file)
    
    # Define your variable names
    vnames = ['llama1', 'llama2', 'alpaca', 'gpt3', 'gpt4o']
    num_vars = len(vnames)
    
    # Create an empty 5x5 array for the edge information
    edge = np.zeros((num_vars, num_vars), dtype=int)
    
    # Create a dictionary to map variable names to indices
    name_to_index = {name: idx for idx, name in enumerate(vnames)}
    
    # Populate the array
    for entry in out_data:
        var1 = entry['variable_name1']
        var2 = entry['variable_name2']
        
        idx1 = name_to_index[var1]
        idx2 = name_to_index[var2]
        
        edge[idx1, idx2] = 100
        edge[idx2, idx1] = 100
    
    # Plot the heatmap in the correct subplot
    ax = axes[i]
    sns.heatmap(edge, annot=True, fmt="d", cmap=cmap, xticklabels=desired_order, yticklabels=desired_order, center=0.5, cbar=False, ax=ax, annot_kws={"size": 10})  # Reduce annotation size
    ax.set_title(f'({chr(65 + i)}) {sub.replace("_", " ").title()}', fontsize=13, pad=20)  # Adjust title font size and padding
    ax.set_xlabel('Model', fontsize=12)
    ax.set_ylabel('Model', fontsize=12)
    ax.tick_params(axis='x', rotation=30, labelsize=10)  # Adjust x label rotation and font size
    ax.tick_params(axis='y', rotation=30, labelsize=10)  # Adjust y label font size

# Adjust the layout and spacing
plt.subplots_adjust(left=0.05, right=0.95, top=0.8, bottom=0.2, wspace=-1)
# Adjust the layout and spacing
plt.tight_layout(pad=3.0, w_pad=4.0)  # Increase pad and w_pad for more space

# Save the figure
output_filename = f'data/out/pic_training_pairs.pdf'
plt.savefig(output_filename, dpi=300,  format='pdf')
plt.show()

print(f"Figure saved as {output_filename}")
